In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import kagglehub
import pandas as pd
import numpy as np
import os

sys.path.append('..')   # ajoute memoirev/src aux dossiers de recherche de les scripts dans src

from src.features import preparer_tables, restreindre_perimetre


/Users/leonardpatron/memory/football-key-person-insurance/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = kagglehub.dataset_download("xfkzujqjvx97n/football-datasets")
tables = {}
print("Path to dataset files:", path)
for root, dirs, files in os.walk(path):
    for f in files:
        #print(os.path.relpath(os.path.join(root, f), path))
        if f.endswith(".csv"):
            nom = os.path.splitext(f)[0]
            tables[nom] = pd.read_csv(os.path.join(root, f), low_memory=False)


propres = preparer_tables(tables)        # tables nettoyées, toutes les compétitions
etude = restreindre_perimetre(propres)



Path to dataset files: /Users/leonardpatron/.cache/kagglehub/datasets/xfkzujqjvx97n/football-datasets/versions/2
player_profiles                     (92671, 34) -> (92671, 17)
team_details                         (2175, 12) -> (2175, 5)
team_competitions_seasons           (196378, 7) -> (2175, 4)
player_injuries                     (143195, 7) -> (143084, 7)
player_national_performances         (92701, 9) -> (92669, 8)
transfer_history                  (1101440, 10) -> (1101317, 10)
player_market_value                 (901429, 3) -> (901429, 3)
player_performances               (1878719, 20) -> (1878719, 19)
player_teammates_played_with       (1257342, 6) -> (1257342, 6)
Périmètre : 25084 joueurs, 442 clubs, 110977 joueurs-saisons

  Premier League (Angleterre)             4286 joueurs
  LaLiga (Espagne)                        4147 joueurs
  Serie A (Italie)                        4540 joueurs
  Bundesliga (Allemagne)                  3353 joueurs
  Ligue 1 (France)                    

### Market values sur les 10 championnats restreints

In [3]:
valeurs = etude["player_market_value"].copy()

# Vérifier les colonnes disponibles
print(valeurs.columns.tolist())

# Calculer les intervalles entre relevés pour chaque joueur
observations = (
    valeurs.dropna(subset=["player_id", "date_valeur"])
    .drop_duplicates(subset=["player_id", "date_valeur"])
    .sort_values(["player_id", "date_valeur"])
    .copy()
)

observations["ecart_jours"] = (
    observations.groupby("player_id")["date_valeur"]
    .diff()
    .dt.days
)

ecarts = observations["ecart_jours"].dropna()

print("Joueurs avec plusieurs dates :",
      (observations.groupby("player_id")["date_valeur"].nunique() > 1).sum())
print("Intervalles calculés :", len(ecarts))
display(ecarts.describe())

['player_id', 'date_valeur', 'value']
Joueurs avec plusieurs dates : 23456
Intervalles calculés : 392727


count    392727.000000
mean        194.778220
std         196.226755
min           1.000000
25%         116.000000
50%         167.000000
75%         203.000000
max        6008.000000
Name: ecart_jours, dtype: float64

In [4]:
from src.demi_saisons import construire_demi_saisons
from src.performances import agreger_performances

# 1. Table joueur × demi-saison (club, championnat, âge, poste)
demi_saisons = construire_demi_saisons(etude)

# 2. Performances agrégées par joueur × saison
perf_saison = agreger_performances(etude["player_performances"])
perf_saison = perf_saison.rename(columns={"nb_clubs": "nb_clubs_saison"})  # évite le conflit avec nb_clubs de demi_saisons

# 3. Jointure sur (joueur, saison)
base = demi_saisons.merge(perf_saison, on=["player_id", "season_name"],
                          how="left", validate="many_to_one")

# 4. Contrôles
assert len(base) == len(demi_saisons), "la jointure a changé le nombre de lignes"
print("Lignes :", len(base))
print("Sans performances :", base["nb_on_pitch"].isna().sum(),
      f"({base['nb_on_pitch'].isna().mean():.1%})")
display(base.head())

Lignes : 221492
Sans performances : 2381 (1.1%)


,player_id,season_name,moitie,debut,fin,team_id,club_name,jours_club_principal,nb_clubs,competition_id,...,direct_red_cards,penalty_goals,goals_conceded,clean_sheets,nb_competitions,nb_clubs_saison,nb_titularisations,nb_matchs_complets,part_titulaire,minutes_estimees
0,1,00/01,1,2000-07-01,2000-12-31,2,1.FC Kaiserslautern,184,1,L1,...,2.0,0.0,0.0,0.0,4.0,2.0,23.0,17.0,0.821429,2045.0
1,1,00/01,2,2001-01-01,2001-06-30,2,1.FC Kaiserslautern,181,1,L1,...,2.0,0.0,0.0,0.0,4.0,2.0,23.0,17.0,0.821429,2045.0
2,1,01/02,1,2001-07-01,2001-12-31,2,1.FC Kaiserslautern,184,1,L1,...,0.0,1.0,0.0,0.0,3.0,2.0,28.0,22.0,0.933333,2420.0
3,1,01/02,2,2002-01-01,2002-06-30,2,1.FC Kaiserslautern,181,1,L1,...,0.0,1.0,0.0,0.0,3.0,2.0,28.0,22.0,0.933333,2420.0
4,1,02/03,1,2002-07-01,2002-12-31,2,1.FC Kaiserslautern,184,1,L1,...,0.0,2.0,0.0,0.0,3.0,2.0,30.0,21.0,0.882353,2575.0
